# Solar Filament Segmentation 2026 — PQ-optimised pipeline

**Idea in one line:** predict a *calibrated* filament probability map (plus a spine map) at full-disk context, then turn it into instances with a decoder whose every threshold is **tuned directly on Panoptic Quality** on out-of-fold images.

| Step | What | Why it matters for PQ |
|---|---|---|
| 1 | Limb-darkening flattening + CLAHE, 3-channel input | same contrast for every observatory (Bh, Mh, Lh, Ch, Th, Uh) |
| 2 | **Soft targets** = fraction of annotators that marked each pixel | the model learns the *consensus* polygon, not one annotator's noise |
| 3 | Split **by observation day** | images of the same day show the same filaments → otherwise CV leaks and over-estimates |
| 4 | U-Net on the **whole disk at 1024**, 2 heads: filament + spine | global context separates filaments from sunspots / plage; spine head teaches continuity |
| 5 | Dihedral TTA, fold ensemble, probabilities upsampled to 2048 | smoother, better-calibrated boundaries → higher IoU per match |
| 6 | **Hysteresis + spine-bridge instance decoder**, per-instance closing / hole filling / dilation, overlap removal | fixes fragmentation (one-to-many) and keeps polygons solid like the annotations |
| 7 | **Grid-search of all decoder params + instance score / area filter on OOF PQ** | each FP costs 0.5 in the denominator; low-confidence blobs are removed only when it pays |
| 8 | Inter-annotator PQ = realistic ceiling | tells you how far above 0.4–0.5 you can actually go |

## v2 — what is new

* **Runs on Colab (Drive), Kaggle, Mac or CPU**; low RAM (no worker processes, compact OOF storage).
* **Resumes** an interrupted fold from its last epoch; already-trained folds are skipped → just Run All again after a disconnect.
* Same encoder / resolution / folds as v1, so your existing `model_fold0.pth` is reused and ensembled with the new folds.
* Wider decoder search (v1 picked values at the edge of its grid).
* **Instance re-scoring**: a small gradient-boosting model predicts, for every candidate filament, the chance that annotators drew it — using shape, size, position and probability features. It is cross-validated by day and only used if it beats the plain decoder on OOF PQ.

**Typical use:** `TRAIN_FOLDS = [1]` (≈ 2 h on Colab GPU) → Run All → next session `[2]` → … Every run ends with a fresh `submission.csv` using all folds trained so far.

In [ ]:
try:                                   # Google Colab: mount Drive (skipped elsewhere)
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
!pip install -q segmentation-models-pytorch pycocotools

In [ ]:
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')   # run any op MPS lacks on CPU instead of crashing
import json, glob, math, random, time, gc, copy, itertools, threading
from concurrent.futures import ThreadPoolExecutor
from multiprocessing.pool import ThreadPool
from collections import defaultdict
import numpy as np, pandas as pd, cv2
from scipy import ndimage as ndi
import torch, torch.nn as nn, torch.nn.functional as F
from pycocotools import mask as mask_utils
import segmentation_models_pytorch as smp

cv2.setNumThreads(2)

class CFG:
    DATA_ROOT = '/content/drive/MyDrive/Solar Filament Segmentation Challenge 2026/MAGFiLO_1.0_Kaggle_2026'
    OUT = '/content/drive/MyDrive/Solar Filament Segmentation Challenge 2026/working'
    WEIGHTS_DIRS = [OUT]               # folders searched for model_fold*.pth (fold 0 from v1 is already there)
    IMG_SIZE = 1024                    # keep identical to the run that trained fold 0
    ARCH = 'Unet'
    ENCODER = 'efficientnet-b4'        # keep identical to fold 0, otherwise its weights cannot be loaded
    ENCODER_WEIGHTS = 'imagenet'
    N_FOLDS = 5
    TRAIN_FOLDS = [1]                  # next sessions: [2], [3], [4] (done folds are skipped, [] = only predict)
    EPOCHS = 45                        # v1 validation PQ plateaued after ~40 epochs
    BATCH = 4
    ACCUM = 2
    LR = 3e-4
    WD = 1e-4
    WARMUP_EPOCHS = 1
    EMA_DECAY = 0.995
    SPINE_W = 0.5
    SPINE_THICK = 3                    # px at IMG_SIZE
    VAL_EVERY = 3
    LOADER_THREADS = 4                 # threads preparing augmented batches (no worker processes -> low RAM, Mac-safe)
    AMP = True                         # mixed precision on CUDA only; MPS runs in float32 (more stable)
    TTA = 8                            # 1, 2, 4 or 8 dihedral transforms
    SEED = 42
    TUNE_MAX_IMAGES = 300              # OOF images (random sample) used for decoder tuning and re-scoring
    N_PROC = 4                         # threads for decoder tuning
    ALWAYS_ONE_ROW = True              # emit the best instance even if an image has none above threshold

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(CFG.SEED)
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
USE_AMP = CFG.AMP and DEVICE == 'cuda'

def free_gpu():
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    elif DEVICE == 'mps':
        torch.mps.empty_cache()

def find_data_root():
    cands = [CFG.DATA_ROOT, '.', '..', '/kaggle/input', '/content/drive/MyDrive']
    for c in cands:
        if c and os.path.isdir(c):
            js = glob.glob(os.path.join(c, '**', '*Annotations*train*.json'), recursive=True)
            if js:
                return os.path.dirname(os.path.dirname(js[0]))
    raise FileNotFoundError('MAGFiLO train json not found - set CFG.DATA_ROOT to the MAGFiLO_1.0_Kaggle_2026 folder')

CFG.DATA_ROOT = find_data_root()
if CFG.DATA_ROOT.startswith('/kaggle/input'):
    CFG.OUT = '/kaggle/working'; CFG.WEIGHTS_DIRS = ['/kaggle/working'] + CFG.WEIGHTS_DIRS
TRAIN_JSON = glob.glob(os.path.join(CFG.DATA_ROOT, 'train', '*.json'))[0]
TRAIN_DIR = os.path.join(CFG.DATA_ROOT, 'train', 'train_images')
TEST_DIR = os.path.join(CFG.DATA_ROOT, 'test', 'test_images')
os.makedirs(CFG.OUT, exist_ok=True)
print('device:', DEVICE, '| torch', torch.__version__, '| data:', CFG.DATA_ROOT)

## 1. Annotations → per-image groups of annotators

In [ ]:
def stem_of(name):
    s = os.path.splitext(os.path.basename(name))[0]
    return s.split('-')[-1]          # '010401-20160920230134Lh' -> '20160920230134Lh'

def list_images(d):
    out = {}
    for ext in ('*.jpeg', '*.jpg', '*.png', '*.JPEG', '*.JPG'):
        for p in glob.glob(os.path.join(d, ext)):
            out[stem_of(p)] = p
    return out

coco = json.load(open(TRAIN_JSON))
train_files = list_images(TRAIN_DIR)
anns_by_img = defaultdict(list)
for a in coco['annotations']:
    anns_by_img[str(a['image_id'])].append(a)

groups = defaultdict(list)            # stem -> list of annotator annotation-sets
img_hw = {}
for im in coco['images']:
    st = stem_of(im['file_name'])
    if st not in train_files:
        continue
    groups[st].append(anns_by_img.get(str(im['id']), []))
    img_hw[st] = (im.get('height', 2048), im.get('width', 2048))

STEMS = sorted(groups)
n_ann = [len(groups[s]) for s in STEMS]
n_fil = [len(a) for s in STEMS for a in groups[s]]
print(f'{len(STEMS)} unique train images, {sum(n_ann)} annotator sets, '
      f'annotators/image {np.bincount(n_ann)}, filaments/set mean {np.mean(n_fil):.1f}')
print('test images:', len(list_images(TEST_DIR)))

In [ ]:
def poly_list(ann):
    seg = ann['segmentation']
    if isinstance(seg, dict):           # already RLE
        return None
    if len(seg) and not isinstance(seg[0], (list, tuple)):
        seg = [seg]
    return [p for p in seg if len(p) >= 6]

def ann_mask(ann, h, w, scale=1.0):
    if isinstance(ann['segmentation'], dict):
        m = mask_utils.decode(ann['segmentation'])
        if scale != 1.0:
            m = cv2.resize(m, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_NEAREST)
        return m.astype(bool)
    polys = [list(np.asarray(p, dtype=float) * scale) for p in poly_list(ann)]
    H, W = round(h * scale), round(w * scale)
    if not polys:
        return np.zeros((H, W), bool)
    rles = mask_utils.frPyObjects(polys, H, W)
    return mask_utils.decode(mask_utils.merge(rles)).astype(bool)

def spine_points(ann):
    sp = ann.get('spine') or []
    if len(sp) and isinstance(sp[0], (list, tuple)):
        sp = [v for part in sp for v in part]
    if len(sp) < 4:
        return None
    return np.asarray(sp, dtype=float).reshape(-1, 2)

def to_local(m):
    # (y0, x0, local bool mask) - compact instance representation used everywhere
    ys, xs = np.where(m)
    if len(ys) == 0:
        return None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    return (int(y0), int(x0), m[y0:y1, x0:x1].copy())

def gt_instances(ann_set, h, w, scale=1.0):
    out = []
    for a in ann_set:
        li = to_local(ann_mask(a, h, w, scale))
        if li is not None:
            out.append(li)
    return out

## 2. Preprocessing: disk detection, limb-darkening flattening, CLAHE

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16))

def find_disk(img):
    small = cv2.resize(img, (512, 512), interpolation=cv2.INTER_AREA)
    blur = cv2.GaussianBlur(small, (0, 0), 3)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, stats, cents = cv2.connectedComponentsWithStats(th)
    if n <= 1:
        h, w = img.shape
        return w / 2, h / 2, min(h, w) * 0.45
    k = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    filled = ndi.binary_fill_holes(lab == k)
    s = img.shape[1] / 512.0
    ys, xs = np.nonzero(filled)
    return xs.mean() * s, ys.mean() * s, math.sqrt(filled.sum() / math.pi) * s

def radius_map(shape, cx, cy):
    h, w = shape
    xx = np.arange(w, dtype=np.float32) - np.float32(cx)
    yy = np.arange(h, dtype=np.float32) - np.float32(cy)
    return np.sqrt(yy[:, None] ** 2 + xx[None, :] ** 2)      # float32, no big int64 grids

def disk_mask(shape, dp):
    cx, cy, r = dp
    return radius_map(shape, cx, cy) <= r * 1.005

def preprocess(img):
    # img: uint8 HxW -> (uint8 HxWx3 input, bool disk mask, (cx, cy, r))
    h, w = img.shape
    cx, cy, r = find_disk(img)
    rr = radius_map(img.shape, cx, cy)
    disk = rr <= r * 1.005
    ri = np.minimum(rr, r * 1.2).astype(np.int32)
    f = img.astype(np.float32)
    cnt = np.bincount(ri[disk], minlength=ri.max() + 1)
    sm = np.bincount(ri[disk], weights=f[disk], minlength=ri.max() + 1)
    prof = ndi.gaussian_filter1d(sm / np.maximum(cnt, 1), 5)
    last = max(int(r) - 2, 1)
    prof[last:] = prof[last]
    flat = f / np.maximum(prof[ri], 1.0)
    flat = np.clip((flat - 0.55) / 0.9, 0, 1)          # ~[0.55,1.45] -> [0,1]
    flat_u8 = (flat * 255).astype(np.uint8)
    flat_u8[~disk] = 0
    cl = _clahe.apply(flat_u8); cl[~disk] = 0
    raw = img.copy(); raw[~disk] = 0
    return np.dstack([raw, flat_u8, cl]), disk, (cx, cy, r)

def load_input(path, size):
    # -> (uint8 size x size x 3, disk params (cx, cy, r) at full res, full-res shape)
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f'cannot read {path}')
    x, _, dp = preprocess(img)
    xs = cv2.resize(x, (size, size), interpolation=cv2.INTER_AREA)
    return xs, dp, img.shape

## 3. Targets: soft consensus mask + spine map, cached at model resolution

In [ ]:
def build_targets(stem, size):
    h, w = img_hw[stem]
    sc = size / w
    fg = np.zeros((size, size), np.float32)
    sp = np.zeros((size, size), np.float32)
    sets = groups[stem]
    for anns in sets:
        u = np.zeros((size, size), bool)
        s = np.zeros((size, size), np.uint8)
        for a in anns:
            u |= ann_mask(a, h, w, sc)
            pts = spine_points(a)
            if pts is not None:
                cv2.polylines(s, [np.round(pts * sc).astype(np.int32)], False, 1, CFG.SPINE_THICK)
        fg += u; sp += s
    return fg / len(sets), sp / len(sets)

def _cache_one(st):
    x, _, _ = load_input(train_files[st], CFG.IMG_SIZE)
    fg, sp = build_targets(st, CFG.IMG_SIZE)
    return st, (x, (fg * 255).astype(np.uint8), (sp * 255).astype(np.uint8))

t0 = time.time()
with ThreadPool(CFG.N_PROC) as tp:
    CACHE = dict(tp.map(_cache_one, STEMS))
mb = sum(sum(a.nbytes for a in v) for v in CACHE.values()) / 2**20
print(f'cached {len(CACHE)} images ({mb:.0f} MB) in {time.time()-t0:.0f}s')

## 4. Folds grouped by observation day

In [ ]:
def make_folds(stems, n_folds, seed):
    day = defaultdict(list)
    for s in stems:
        day[s[:8]].append(s)
    days = list(day); random.Random(seed).shuffle(days)
    days.sort(key=lambda d: -len(day[d]))
    load = [0] * n_folds; fold = {}
    for d in days:
        k = int(np.argmin(load))
        for s in day[d]:
            fold[s] = k
        load[k] += len(day[d])
    return fold

FOLD = make_folds(STEMS, CFG.N_FOLDS, CFG.SEED)
print('images per fold:', np.bincount(list(FOLD.values())))

## 5. Dataset + augmentation (rotation-invariant Sun → any angle)

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

def augment(x, fg, sp):
    n = x.shape[0]
    if random.random() < 0.5:
        x, fg, sp = x[:, ::-1], fg[:, ::-1], sp[:, ::-1]
    k = random.randint(0, 3)
    x, fg, sp = np.rot90(x, k), np.rot90(fg, k), np.rot90(sp, k)
    if random.random() < 0.8:
        ang = random.uniform(-180, 180); scl = random.uniform(0.9, 1.1)
        c = n / 2 + random.uniform(-0.03, 0.03) * n
        M = cv2.getRotationMatrix2D((c, c), ang, scl)
        x = cv2.warpAffine(np.ascontiguousarray(x), M, (n, n), flags=cv2.INTER_LINEAR, borderValue=0)
        fg = cv2.warpAffine(np.ascontiguousarray(fg), M, (n, n), flags=cv2.INTER_LINEAR, borderValue=0)
        sp = cv2.warpAffine(np.ascontiguousarray(sp), M, (n, n), flags=cv2.INTER_LINEAR, borderValue=0)
    x = np.ascontiguousarray(x).astype(np.float32) / 255.0
    if random.random() < 0.8:                      # contrast / brightness / gamma
        a = random.uniform(0.75, 1.3); b = random.uniform(-0.1, 0.1)
        g = random.uniform(0.75, 1.35)
        x = np.clip(np.clip(x * a + b, 0, 1) ** g, 0, 1)
    if random.random() < 0.3:                      # seeing blur
        x = cv2.GaussianBlur(x, (0, 0), random.uniform(0.5, 1.5))
    if random.random() < 0.3:
        x = np.clip(x + np.random.normal(0, random.uniform(0.005, 0.03), x.shape).astype(np.float32), 0, 1)
    return x, np.ascontiguousarray(fg), np.ascontiguousarray(sp)

class FilDS(torch.utils.data.Dataset):
    def __init__(self, stems, train):
        self.stems, self.train = stems, train
    def __len__(self):
        return len(self.stems)
    def __getitem__(self, i):
        x, fg, sp = CACHE[self.stems[i]]
        if self.train:
            x, fg, sp = augment(x, fg, sp)
        else:
            x = x.astype(np.float32) / 255.0
        x = (x - MEAN) / STD
        return (torch.from_numpy(x.transpose(2, 0, 1).copy()),
                torch.from_numpy(fg.astype(np.float32) / 255.0),
                torch.from_numpy(sp.astype(np.float32) / 255.0))

class ThreadLoader:
    # Batches prepared by threads (OpenCV/NumPy release the GIL). Unlike DataLoader worker processes,
    # this never copies the image cache into child processes and is safe in Jupyter on macOS.
    def __init__(self, ds, batch, shuffle=True, drop_last=True, threads=4, prefetch=2):
        self.ds, self.b, self.shuffle, self.drop_last = ds, batch, shuffle, drop_last
        self.threads, self.prefetch = threads, prefetch
    def __len__(self):
        n = len(self.ds)
        return n // self.b if self.drop_last else math.ceil(n / self.b)
    def _make(self, idx):
        items = [self.ds[i] for i in idx]
        return tuple(torch.stack(t) for t in zip(*items))
    def __iter__(self):
        order = np.random.permutation(len(self.ds)) if self.shuffle else np.arange(len(self.ds))
        batches = [order[i:i + self.b] for i in range(0, len(order), self.b)]
        if self.drop_last and batches and len(batches[-1]) < self.b:
            batches = batches[:-1]
        with ThreadPoolExecutor(self.threads) as ex:
            futs = [ex.submit(self._make, b) for b in batches[:self.threads * self.prefetch]]
            nxt = len(futs)
            for i in range(len(batches)):
                out = futs[i].result(); futs[i] = None
                if nxt < len(batches):
                    futs.append(ex.submit(self._make, batches[nxt])); nxt += 1
                yield out

## 6. Model, loss, EMA

In [ ]:
def build_model(pretrained=True):
    kw = dict(encoder_name=CFG.ENCODER, encoder_weights=CFG.ENCODER_WEIGHTS if pretrained else None,
              in_channels=3, classes=2)
    if CFG.ARCH == 'Unet':
        kw['decoder_attention_type'] = 'scse'
    return smp.create_model(CFG.ARCH, **kw)

def soft_dice_loss(logit, t, eps=1.0):
    p = torch.sigmoid(logit)
    inter = (p * t).sum((1, 2)); den = (p + t).sum((1, 2))
    return (1 - (2 * inter + eps) / (den + eps)).mean()

def loss_fn(logits, fg, sp):
    l_fg = F.binary_cross_entropy_with_logits(logits[:, 0], fg) + soft_dice_loss(logits[:, 0], fg)
    l_sp = F.binary_cross_entropy_with_logits(logits[:, 1], sp) + soft_dice_loss(logits[:, 1], sp)
    return l_fg + CFG.SPINE_W * l_sp

class EMA:
    def __init__(self, model, decay):
        self.m = copy.deepcopy(model).eval(); self.d = decay; self.n = 0
        for p in self.m.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.d, (1 + self.n) / (10 + self.n))   # warm-up: early weights are not frozen in the average
        msd = model.state_dict()
        for k, v in self.m.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else:
                v.copy_(msd[k])

## 7. Instance decoder + PQ metric

* **grouping mask** = `prob > t_low` ∪ `spine > t_spine` → connected components (fragments of one filament join through faint material or the predicted spine)
* a component is kept only if it contains a **core** pixel `prob > t_high` (hysteresis)
* instance pixels = component ∩ `prob > t_mask`, then per-instance closing, hole filling, optional dilation
* overlaps resolved by score; final filter on **area** and **mean probability** — all tuned on PQ

In [ ]:
DEFAULT_P = dict(t_low=0.35, t_high=0.6, t_mask=0.45, t_spine=1.1, close=5, dilate=0, min_area=150, min_score=0.55)
_kern = {}
def kern(r):
    if r not in _kern:
        _kern[r] = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))
    return _kern[r]

def decode(fg, sp, p, disk=None):
    # fg, sp: float32 [0,1] HxW. Returns list of (y0, x0, mask, area, score) sorted by score desc.
    if disk is not None:
        fg = fg * disk
    grp = fg > p['t_low']
    if p['t_spine'] <= 1.0:
        grp |= (sp > p['t_spine']) & (fg > p['t_low'] * 0.5)
    lab, n = ndi.label(grp, structure=np.ones((3, 3)))
    if n == 0:
        return []
    core = np.bincount(lab[fg > p['t_high']], minlength=n + 1) > 0
    H, W = fg.shape
    pad = p['close'] + p['dilate'] + 1
    cands = []
    for i, sl in enumerate(ndi.find_objects(lab), 1):
        if sl is None or not core[i]:
            continue
        y0 = max(sl[0].start - pad, 0); y1 = min(sl[0].stop + pad, H)
        x0 = max(sl[1].start - pad, 0); x1 = min(sl[1].stop + pad, W)
        f = fg[y0:y1, x0:x1]
        m = (lab[y0:y1, x0:x1] == i) & (f > p['t_mask'])
        if not m.any():
            continue
        if p['close'] > 0:
            m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_CLOSE, kern(p['close'])).astype(bool)
        m = ndi.binary_fill_holes(m)
        if p['dilate'] > 0:
            m = cv2.dilate(m.astype(np.uint8), kern(p['dilate'])).astype(bool)
        cands.append([y0, x0, m, float(f[m].mean())])
    cands.sort(key=lambda c: -c[3])
    taken = np.zeros((H, W), bool); out = []
    for y0, x0, m, sc in cands:
        h, w = m.shape
        m = m & ~taken[y0:y0 + h, x0:x0 + w]
        a = int(m.sum())
        if a == 0:
            continue
        taken[y0:y0 + h, x0:x0 + w] |= m
        out.append((y0, x0, m, a, sc))
    return out

def filter_inst(inst, min_area, min_score):
    return [c for c in inst if c[3] >= min_area and c[4] >= min_score]

def iou_local(a, b):
    ay, ax, am = a[0], a[1], a[2]; by, bx, bm = b[0], b[1], b[2]
    y0, x0 = max(ay, by), max(ax, bx)
    y1 = min(ay + am.shape[0], by + bm.shape[0]); x1 = min(ax + am.shape[1], bx + bm.shape[1])
    if y1 <= y0 or x1 <= x0:
        return 0.0
    inter = np.logical_and(am[y0 - ay:y1 - ay, x0 - ax:x1 - ax], bm[y0 - by:y1 - by, x0 - bx:x1 - bx]).sum()
    if inter == 0:
        return 0.0
    return inter / (am.sum() + bm.sum() - inter)

def pq_stats(pred, gt):
    # returns (sum IoU of TP, TP, FP, FN); match = IoU > 0.5 (unique by construction)
    pairs = []
    for i, p in enumerate(pred):
        for j, g in enumerate(gt):
            v = iou_local(p, g)
            if v > 0.5:
                pairs.append((v, i, j))
    pairs.sort(reverse=True); up, ug = set(), set(); s = 0.0
    for v, i, j in pairs:
        if i in up or j in ug:
            continue
        up.add(i); ug.add(j); s += v
    tp = len(up)
    return np.array([s, tp, len(pred) - tp, len(gt) - tp], np.float64)

def inst_match(inst, gts):
    # per candidate (predictions are disjoint, so an IoU>0.5 match is unique and PQ is additive over candidates):
    # summed IoU and number of annotator sets in which it is a true positive (IoU > 0.5)
    S = np.zeros(len(inst)); T = np.zeros(len(inst))
    for g in gts:
        for i, c in enumerate(inst):
            b = max((iou_local(c, q) for q in g), default=0.0)
            if b > 0.5:
                S[i] += b; T[i] += 1
    return S, T

def pq_from(st):
    s, tp, fp, fn = st
    den = tp + 0.5 * fp + 0.5 * fn
    return s / den if den > 0 else 0.0, (s / tp if tp else 0.0), (tp / den if den else 0.0)

## 8. Inference helpers (dihedral TTA)

In [ ]:
def to_tensor(x_u8):
    x = (x_u8.astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(x.transpose(2, 0, 1).copy())

@torch.no_grad()
def predict(models, x_u8, tta=CFG.TTA):
    # x_u8: HxWx3 at model size -> (fg, sp) float32 at model size, averaged over models x TTA
    x = to_tensor(x_u8)[None].to(DEVICE)
    tfs = [(f, k) for f in (False, True) for k in range(4)][:tta] if tta > 1 else [(False, 0)]
    batch = []
    for f, k in tfs:
        t = torch.flip(x, [3]) if f else x
        batch.append(torch.rot90(t, k, [2, 3]))
    batch = torch.cat(batch)
    acc = 0
    for m in models:
        with torch.autocast(device_type='cuda' if DEVICE == 'cuda' else 'cpu', enabled=USE_AMP):
            out = torch.sigmoid(m(batch).float())
        for i, (f, k) in enumerate(tfs):
            o = torch.rot90(out[i:i + 1], -k, [2, 3])
            acc = acc + (torch.flip(o, [3]) if f else o)
    acc = (acc / (len(models) * len(tfs)))[0].cpu().numpy()
    return acc[0], acc[1]

def upsample(p, shape):
    return cv2.resize(p, (shape[1], shape[0]), interpolation=cv2.INTER_LINEAR)

## 9. Training (checkpoint chosen by validation PQ, not by loss)

In [ ]:
def quick_val_pq(model, val_stems):
    # PQ at model resolution, default decoder - used only to pick the best epoch
    model.eval(); tot = np.zeros(4)
    sc = CFG.IMG_SIZE / 2048.0
    p = dict(DEFAULT_P); p['close'] = max(1, round(p['close'] * sc)); p['min_area'] = p['min_area'] * sc * sc
    for st in val_stems:
        fg, sp = predict([model], CACHE[st][0], tta=1)
        inst = filter_inst(decode(fg, sp, p), p['min_area'], p['min_score'])
        for anns in groups[st]:
            tot += pq_stats(inst, VAL_GT_SMALL[st][id(anns)])
    return pq_from(tot)

def train_fold(fold):
    tr = [s for s in STEMS if FOLD[s] != fold]; va = [s for s in STEMS if FOLD[s] == fold]
    global VAL_GT_SMALL
    VAL_GT_SMALL = {}
    for st in va:
        h, w = img_hw[st]
        VAL_GT_SMALL[st] = {id(a): gt_instances(a, h, w, CFG.IMG_SIZE / w) for a in groups[st]}
    dl = ThreadLoader(FilDS(tr, True), CFG.BATCH, shuffle=True, drop_last=True, threads=CFG.LOADER_THREADS)
    model = build_model(True)
    ema = EMA(model, CFG.EMA_DECAY)
    path = os.path.join(CFG.OUT, f'model_fold{fold}.pth')
    last = os.path.join(CFG.OUT, f'last_fold{fold}.pth')      # resume point if the run is interrupted
    model.to(DEVICE); ema.m.to(DEVICE)
    net = nn.DataParallel(model) if DEVICE == 'cuda' and torch.cuda.device_count() > 1 else model
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WD)
    per_ep = max(1, len(dl) // CFG.ACCUM)
    steps = CFG.EPOCHS * per_ep; warm = CFG.WARMUP_EPOCHS * per_ep
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, warm)) *
                                              0.5 * (1 + math.cos(math.pi * min(1.0, s / max(1, steps)))))
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best, start = -1.0, 0
    if os.path.exists(last):
        ck = torch.load(last, map_location='cpu', weights_only=False)   # our own checkpoint file
        model.load_state_dict(ck['model']); ema.m.load_state_dict(ck['ema']); ema.n = ck['ema_n']
        opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        best, start = ck['best'], ck['epoch']
        print(f'resuming fold {fold} from epoch {start} (best PQ so far {best:.4f})')
    t_fold = time.time()
    for ep in range(start, CFG.EPOCHS):
        net.train(); t0 = time.time(); tl = 0; opt.zero_grad()
        for it, (x, fg, sp) in enumerate(dl):
            x, fg, sp = x.to(DEVICE), fg.to(DEVICE), sp.to(DEVICE)
            with torch.autocast(device_type='cuda' if DEVICE == 'cuda' else 'cpu', enabled=USE_AMP):
                logits = net(x)
            loss = loss_fn(logits.float(), fg, sp)
            scaler.scale(loss / CFG.ACCUM).backward(); tl += loss.item()
            if (it + 1) % CFG.ACCUM == 0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step(); ema.update(model)
        msg = f'fold {fold} ep {ep+1}/{CFG.EPOCHS} loss {tl/max(1,len(dl)):.4f} {time.time()-t0:.0f}s'
        if (ep + 1) % CFG.VAL_EVERY == 0 or ep + 1 == CFG.EPOCHS:
            pq, sq, rq = quick_val_pq(ema.m, va)
            msg += f' | val PQ {pq:.4f} (SQ {sq:.3f} RQ {rq:.3f})'
            if pq > best:
                best = float(pq); torch.save({k: v.detach().cpu() for k, v in ema.m.state_dict().items()}, path); msg += ' *'
        if ep == start:
            eta = (time.time() - t_fold) * (CFG.EPOCHS - start)
            msg += f'  [estimated time for this fold: {eta/3600:.1f} h]'
        print(msg, flush=True)
        torch.save({'model': model.state_dict(), 'ema': ema.m.state_dict(), 'ema_n': ema.n, 'opt': opt.state_dict(),
                    'sched': sched.state_dict(), 'best': float(best), 'epoch': ep + 1}, last)
    if os.path.exists(last):
        os.remove(last)                                         # fold finished; best weights are in model_fold*.pth
    del model, ema, net, opt; free_gpu()
    return best

for f in CFG.TRAIN_FOLDS:
    done = any(os.path.exists(os.path.join(d, f'model_fold{f}.pth')) for d in CFG.WEIGHTS_DIRS)
    if done and not os.path.exists(os.path.join(CFG.OUT, f'last_fold{f}.pth')):
        print(f'fold {f}: trained weights found, skipping (delete outputs/model_fold{f}.pth to retrain)'); continue
    print('best quick PQ fold', f, '=', train_fold(f))

## 10. Load all available fold models

In [ ]:
def find_weights(f):
    for d in CFG.WEIGHTS_DIRS:
        p = os.path.join(d, f'model_fold{f}.pth')
        if os.path.exists(p):
            return p
    return None

MODELS = {}
for f in range(CFG.N_FOLDS):
    p = find_weights(f)
    if p:
        m = build_model(False); m.load_state_dict(torch.load(p, map_location='cpu')); MODELS[f] = m.to(DEVICE).eval()
print('folds available:', sorted(MODELS))

## 11. Out-of-fold predictions at 2048 (TTA) + realistic ceiling (annotator vs annotator)

In [ ]:
CACHE = {}; gc.collect()     # training cache no longer needed (re-run section 3 before training again)

# Low-RAM storage: probabilities kept at model resolution (uint8) and upsampled to 2048 only when decoding.
OOF = {}   # stem -> (fg uint8, sp uint8, disk params, full shape, list of GT annotator sets at full res)
oof_stems = [s for s in STEMS if FOLD[s] in MODELS]
oof_stems = sorted(random.Random(CFG.SEED).sample(oof_stems, min(len(oof_stems), CFG.TUNE_MAX_IMAGES)))
t0 = time.time()
for st in oof_stems:
    x, dp, shape = load_input(train_files[st], CFG.IMG_SIZE)
    fg, sp = predict([MODELS[FOLD[st]]], x)
    h, w = img_hw[st]
    OOF[st] = ((fg * 255).astype(np.uint8), (sp * 255).astype(np.uint8), dp, shape,
               [gt_instances(a, h, w) for a in groups[st]])
print(f'OOF on {len(OOF)} images in {time.time()-t0:.0f}s')

def oof_maps(st):
    fgu, spu, dp, shape, gts = OOF[st]
    fg = upsample(fgu.astype(np.float32) / 255.0, shape)
    sp = upsample(spu.astype(np.float32) / 255.0, shape)
    return fg, sp, disk_mask(shape, dp), gts

# Ceiling: score annotator A against annotator B on the same image
tot = np.zeros(4)
for st in STEMS[:200]:
    sets = groups[st]
    if len(sets) < 2:
        continue
    h, w = img_hw[st]
    gi = [gt_instances(a, h, w) for a in sets]
    for i in range(len(gi)):
        for j in range(len(gi)):
            if i != j:
                tot += pq_stats(gi[i], gi[j])
print('INTER-ANNOTATOR PQ (human ceiling) = %.4f  SQ %.3f  RQ %.3f' % pq_from(tot))

## 12. Tune every decoder parameter directly on OOF PQ (coordinate search)

In [ ]:
def _eval_image(args):
    # decode once, match once, then score every (min_area, min_score) filter combination in one shot
    st, p = args
    fg, sp, disk, gts = oof_maps(st)
    inst = decode(fg, sp, p, disk)
    n_gt = sum(len(g) for g in gts); ns = len(gts)
    out = np.zeros((len(AREAS) * len(SCORES), 4)); out[:, 3] = n_gt
    if not inst:
        return out
    S, T = inst_match(inst, gts)
    a = np.array([c[3] for c in inst]); sc = np.array([c[4] for c in inst])
    for k, (ma, ms) in enumerate(itertools.product(AREAS, SCORES)):
        keep = (a >= ma) & (sc >= ms)
        tp = T[keep].sum()
        out[k] = (S[keep].sum(), tp, (ns - T[keep]).sum(), n_gt - tp)
    return out

AREAS = [0, 50, 100, 150, 200, 300, 400, 600, 800, 1200, 1500, 2500]
SCORES = [0.0, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.83, 0.86, 0.89, 0.92]

def evaluate(p):
    # returns best (pq, min_area, min_score) for structural params p (filters searched exhaustively)
    jobs = [(st, p) for st in OOF]
    with ThreadPool(CFG.N_PROC) as pool:          # threads: no copies of the data, safe on macOS
        res = pool.map(_eval_image, jobs)
    tot = np.sum(res, axis=0)
    pqs = [pq_from(t)[0] for t in tot]
    k = int(np.argmax(pqs))
    return pqs[k], AREAS[k // len(SCORES)], SCORES[k % len(SCORES)], pq_from(tot[k])

SEARCH = dict(t_high=[0.4, 0.5, 0.6, 0.7, 0.8, 0.9], t_low=[0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5],
              t_mask=[0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65], t_spine=[1.1, 0.2, 0.3, 0.4, 0.5, 0.7],
              close=[0, 3, 6, 8, 10, 14], dilate=[0, 1, 2, 3, 4])
# v1 result on fold 0 as the starting point (saves time); the search moves away from it if needed
best_p = dict(DEFAULT_P, t_low=0.2, t_spine=0.5, close=6)
best_score, a, s, det = evaluate(best_p); best_p.update(min_area=a, min_score=s)
print(f'start PQ {best_score:.4f}', best_p)
t_tune = time.time()
for rnd in range(2):
    improved = False
    for key, vals in SEARCH.items():
        for v in vals:
            if v == best_p[key]:
                continue
            p = dict(best_p); p[key] = v
            if p['t_low'] > p['t_high']:
                continue
            sc, a, s, det = evaluate(p)
            if sc > best_score + 1e-5:
                best_score = sc; best_p = dict(p, min_area=a, min_score=s); improved = True
                print(f'  {key}={v}: PQ {sc:.4f} (SQ {det[1]:.3f} RQ {det[2]:.3f}) area>={a} score>={s}', flush=True)
    print(f'round {rnd+1}: PQ {best_score:.4f} ({(time.time()-t_tune)/60:.1f} min)', best_p)
    if not improved:
        break
json.dump(best_p, open(os.path.join(CFG.OUT, 'decoder_params.json'), 'w'), indent=1)

## 13. Diagnostics: where is PQ lost?

In [ ]:
tot = np.zeros(4); per = []
o2m = m2o = 0
for st in OOF:
    fg, sp, disk, gts = oof_maps(st)
    inst = filter_inst(decode(fg, sp, best_p, disk), best_p['min_area'], best_p['min_score'])
    for g in gts:
        r = pq_stats(inst, g); tot += r; per.append(pq_from(r)[0])
        # fragmentation (one GT touched by >=2 preds with IoU>0.1) and over-merging
        M = np.array([[iou_local(p, q) for q in g] for p in inst]) if inst and g else np.zeros((0, 0))
        if M.size:
            o2m += int(((M > 0.1).sum(0) >= 2).sum()); m2o += int(((M > 0.1).sum(1) >= 2).sum())
pq, sq, rq = pq_from(tot)
print(f'OOF PQ {pq:.4f} = SQ {sq:.3f} x RQ {rq:.3f} | TP {tot[1]:.0f} FP {tot[2]:.0f} FN {tot[3]:.0f}')
print(f'mean per-image PQ {np.mean(per):.4f} | GT split into >=2 preds: {o2m} | preds covering >=2 GT: {m2o}')

In [ ]:
import matplotlib.pyplot as plt
show = list(OOF)[:3]
fig, axs = plt.subplots(len(show), 3, figsize=(18, 6 * len(show)))
axs = np.atleast_2d(axs)
for r, st in enumerate(show):
    img = cv2.imread(train_files[st], cv2.IMREAD_GRAYSCALE)
    fg, sp, disk, gts = oof_maps(st)
    inst = filter_inst(decode(fg, sp, best_p, disk), best_p['min_area'], best_p['min_score'])
    pm = np.zeros(img.shape, np.int32)
    for k, (y0, x0, m, a, s) in enumerate(inst, 1):
        pm[y0:y0 + m.shape[0], x0:x0 + m.shape[1]][m] = k
    gm = np.zeros(img.shape, np.int32)
    for k, (y0, x0, m) in enumerate(gts[0], 1):
        gm[y0:y0 + m.shape[0], x0:x0 + m.shape[1]][m] = k
    axs[r, 0].imshow(img, cmap='gray'); axs[r, 0].set_title(st)
    axs[r, 1].imshow(img, cmap='gray'); axs[r, 1].imshow(np.ma.masked_equal(gm, 0) % 20, cmap='tab20', alpha=.6); axs[r, 1].set_title('GT (annotator 1)')
    axs[r, 2].imshow(img, cmap='gray'); axs[r, 2].imshow(np.ma.masked_equal(pm, 0) % 20, cmap='tab20', alpha=.6); axs[r, 2].set_title(f'pred ({len(inst)})')
    for a in axs[r]: a.axis('off')
plt.tight_layout(); plt.show()

## 14. Instance re-scoring (second stage)

PQ is limited mostly by **RQ** (wrong / missing filaments), not by mask quality. The decoder ranks candidates only by their
mean probability. Here a gradient-boosting model learns, from OOF candidates, *which annotators actually drew* using
size, shape, elongation, position on the disk, spine and probability statistics. Its scores are cross-fitted by
observation day, a single threshold is tuned on PQ, and it is used for the submission **only if it beats the plain decoder**.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
import pickle

FEATS = ['log_area', 'p_mean', 'p_max', 'p_q25', 'p_q75', 'sp_mean', 'sp_max', 'box_h', 'box_w',
         'elong', 'fill', 'r_norm', 'n_inst', 'rank']

def inst_features(inst, fg, sp, dp):
    cx, cy, r = dp
    rows = []
    for k, (y0, x0, m, a, s) in enumerate(inst):
        h, w = m.shape
        f = fg[y0:y0 + h, x0:x0 + w][m]; q = sp[y0:y0 + h, x0:x0 + w][m]
        mo = cv2.moments(m.astype(np.uint8), binaryImage=True); m00 = max(mo['m00'], 1)
        mu20, mu02, mu11 = mo['mu20'] / m00, mo['mu02'] / m00, mo['mu11'] / m00
        d = math.sqrt(max((mu20 - mu02) ** 2 + 4 * mu11 ** 2, 0))
        l1, l2 = (mu20 + mu02 + d) / 2, max((mu20 + mu02 - d) / 2, 1e-3)
        yc, xc = y0 + mo['m01'] / m00, x0 + mo['m10'] / m00
        q25, q75 = np.percentile(f, [25, 75])
        rows.append([math.log1p(a), f.mean(), f.max(), q25, q75, q.mean(), q.max(), h, w,
                     math.sqrt(max(l1, 1e-3) / l2), a / (h * w), math.hypot(xc - cx, yc - cy) / max(r, 1), len(inst), k])
    return np.array(rows, np.float32).reshape(-1, len(FEATS))

def _collect(st):
    fg, sp, disk, gts = oof_maps(st)
    inst = decode(fg, sp, best_p, disk)
    n_gt = sum(len(g) for g in gts)
    if not inst:
        return None, n_gt
    S, T = inst_match(inst, gts)
    return (inst_features(inst, fg, sp, OOF[st][2]), S, T, np.full(len(inst), len(gts)),
            np.array([c[3] for c in inst]), np.array([c[4] for c in inst]), [st[:8]] * len(inst)), n_gt

t0 = time.time()
with ThreadPool(CFG.N_PROC) as pool:
    res = pool.map(_collect, list(OOF))
GT_TOTAL = sum(n for _, n in res)
parts = [r for r, _ in res if r is not None]
X = np.concatenate([r[0] for r in parts]); S = np.concatenate([r[1] for r in parts]); T = np.concatenate([r[2] for r in parts])
NS = np.concatenate([r[3] for r in parts]); AREA = np.concatenate([r[4] for r in parts]); SCORE = np.concatenate([r[5] for r in parts])
G = np.array(sum((r[6] for r in parts), []))
print(f'{len(X)} candidate filaments from {len(OOF)} OOF images ({time.time()-t0:.0f}s)')

def pq_keep(keep):
    s = S[keep].sum(); tp = T[keep].sum(); fp = (NS[keep] - T[keep]).sum(); fn = GT_TOTAL - tp
    den = tp + 0.5 * fp + 0.5 * fn
    return s / den if den else 0.0

base_pq = pq_keep((AREA >= best_p['min_area']) & (SCORE >= best_p['min_score']))
print(f'plain decoder PQ (recomputed): {base_pq:.4f}')

y = T / NS                                   # fraction of annotators that drew this candidate
def make_reg():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                         min_samples_leaf=20, l2_regularization=1.0, random_state=0)
oof_q = np.zeros(len(y))
n_split = min(5, len(set(G)))
for tr, va in GroupKFold(n_splits=n_split).split(X, y, G):
    oof_q[va] = make_reg().fit(X[tr], y[tr]).predict(X[va])
ths = np.round(np.arange(0.0, 0.9, 0.01), 2)
pqs = np.array([pq_keep(oof_q >= t) for t in ths])
RESCORE_T = float(ths[pqs.argmax()]); resc_pq = float(pqs.max())
USE_RESCORE = resc_pq > base_pq + 0.002
print(f're-scored PQ: {resc_pq:.4f} at threshold {RESCORE_T}  ->  use re-scoring: {USE_RESCORE}')

RESCORER = make_reg().fit(X, y) if USE_RESCORE else None
with open(os.path.join(CFG.OUT, 'rescorer.pkl'), 'wb') as fh:
    pickle.dump(dict(model=RESCORER, threshold=RESCORE_T, use=USE_RESCORE, feats=FEATS, decoder=best_p), fh)

## 15. Test inference → submission.csv

In [ ]:
def rle_counts(y0, x0, m, shape):
    full = np.zeros(shape, np.uint8, order='F')
    full[y0:y0 + m.shape[0], x0:x0 + m.shape[1]] = m
    r = mask_utils.encode(full)
    c = r['counts']
    return c.decode('ascii') if isinstance(c, bytes) else c

def select(inst, fg, sp, dp):
    if not inst:
        return []
    if USE_RESCORE:
        q = RESCORER.predict(inst_features(inst, fg, sp, dp))
        kept = [c for c, v in zip(inst, q) if v >= RESCORE_T]
        best = inst[int(np.argmax(q))]
    else:
        kept = filter_inst(inst, best_p['min_area'], best_p['min_score'])
        best = inst[0]
    if not kept and CFG.ALWAYS_ONE_ROW:
        kept = [best]
    return kept

ens = [MODELS[f] for f in sorted(MODELS)]
print('ensemble of folds', sorted(MODELS), '| re-scoring:', USE_RESCORE)
test_files = list_images(TEST_DIR)
rows = []; t0 = time.time()
for st, path in sorted(test_files.items()):
    x, dp, shape = load_input(path, CFG.IMG_SIZE)
    fgs, sps = predict(ens, x)
    fg, sp = upsample(fgs, shape), upsample(sps, shape)
    kept = select(decode(fg, sp, best_p, disk_mask(shape, dp)), fg, sp, dp)
    for k, (y0, x0, m, a, s) in enumerate(kept, 1):
        rows.append((f'{st}_{k}', rle_counts(y0, x0, m, shape)))
sub = pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])
name = f'submission_folds{"".join(map(str, sorted(MODELS)))}.csv'
sub.to_csv(os.path.join(CFG.OUT, name), index=False)
sub.to_csv(os.path.join(CFG.OUT, 'submission.csv'), index=False)
print(f'{len(test_files)} test images -> {len(sub)} filaments ({len(sub)/max(1,len(test_files)):.1f}/image), '
      f'{time.time()-t0:.0f}s -> {name}')
sub.head()

In [ ]:
# Sanity check: decode a few RLEs back and compare with the masks we meant to write
chk = sub.sample(min(5, len(sub)), random_state=0) if len(sub) else sub
for fid, c in zip(chk.filament_id, chk.segmentation_rle):
    m = mask_utils.decode({'size': [2048, 2048], 'counts': c.encode('ascii')})
    print(fid, 'pixels', int(m.sum()))